# GPT-2 3D Semantic & Mechanistic Concept Map

Trace neuron clusters across token positions and layer depths during a single GPT-2 forward pass,
then automatically decode each cluster into a human-readable semantic concept.

## What this notebook does

1. **Multi-layer + multi-token extraction**: Capture activations at every (token, transformer block) pair.
2. **Feature decoding**: `HypoSpaceAPI.decode()` → top-k `Feature` objects per (token, layer).
3. **3D concept map**: (token × layer × score) scatter, colored by intensity band.
4. **Neuron cluster identification**: Group features by `source_index`; rank by total activation.
5. **Temporal evolution**: Score trajectories and 3D trajectory lines per cluster.
6. **Interactive cluster isolation**: Per-cluster (layer × token) heatmap via dropdown.
7. **Automatic semantic labeling**: `wte` vocabulary scoring → WordNet LCH → SBERT fallback.
8. **UMAP Semantic Atlas**: 2D map of GPT-2's residual-stream concept space.
9. **Semantic × Layer / Token × Semantic heatmaps**: Where and when each concept activates.
10. **Semantic Composition Flow**: Stacked area showing concept proportions across depth.
11. **Co-activation Network**: Which semantic concepts always fire together.
12. **3D Bipartite Semantic Graph**: Token nodes ↔ Semantic cluster nodes, three interactive edge types.

All GPT-2 cells skip gracefully if `nnsight`/`torch` are absent.
Semantic cells skip if `nltk`/`sentence-transformers`/`umap-learn`/`hdbscan` are absent.

In [1]:
# Run once per environment
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
!pip install -q nnsight diskcache plotly pandas ipywidgets transformers
!pip install -q nltk sentence-transformers umap-learn hdbscan scikit-learn

In [2]:
import sys
import os
import warnings
import statistics

for _p in [os.path.abspath(".."), os.path.abspath(".")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from api import HypoSpaceAPI
from core.config import DecoderConfig, RuntimeConfig, GovernanceConfig
from viz.canvas import SemanticCanvas
from diagnostics import run_diagnostics
from interpretability.mechanistic import MechanisticAnalyzer

try:
    from data.nnsight_extractor import NNSightExtractor
    from transformers import AutoTokenizer
    NNSIGHT_AVAILABLE = True
    print("nnsight + transformers available")
except ImportError:
    NNSIGHT_AVAILABLE = False
    print("nnsight not available - GPT-2 cells will be skipped")

try:
    import nltk
    nltk.download("wordnet", quiet=True)
    nltk.download("omw-1.4", quiet=True)
    from nltk.corpus import wordnet as wn
    from sentence_transformers import SentenceTransformer
    import umap as umap_lib
    import hdbscan as hdbscan_lib
    SEMANTIC_AVAILABLE = True
    print("semantic deps available (nltk, sentence-transformers, umap-learn, hdbscan)")
except ImportError:
    SEMANTIC_AVAILABLE = False
    print("semantic deps unavailable - install: nltk sentence-transformers umap-learn hdbscan")

print("Imports complete")

nnsight + transformers available


semantic deps available (nltk, sentence-transformers, umap-learn, hdbscan)
Imports complete


## Configuration

Adjust `INPUT_TEXT`, `TOP_K`, or `TOP_N_CLUSTERS` to explore different sentences or depths.

In [3]:
INPUT_TEXT     = "The quick brown fox jumps"
MODEL_NAME     = "gpt2"
N_LAYERS       = 12
TOP_K          = 8
TOP_N_CLUSTERS = 10
WTE_TOP_N      = 30
SBERT_MODEL    = "all-MiniLM-L6-v2"
EDGE_THRESHOLD = 0.15
CACHE_DIR      = "../.hypo_cache"
VERSION        = "0.1.0"

LAYER_PATHS  = [f"transformer.h.{i}" for i in range(N_LAYERS)]
LAYER_LABELS = [f"h.{i}" for i in range(N_LAYERS)]
LAYER_IDS    = [f"gpt2-h{i}" for i in range(N_LAYERS)]

INTENSITY_COLORS = {
    "high-intensity":   "#e74c3c",
    "medium-intensity": "#f39c12",
    "low-intensity":    "#3498db",
    "unknown":          "#95a5a6",
}
CLUSTER_PALETTE = [
    "#e74c3c","#3498db","#2ecc71","#f39c12","#9b59b6",
    "#1abc9c","#e67e22","#34495e","#e91e63","#00bcd4",
]

if NNSIGHT_AVAILABLE:
    _tok_preview  = AutoTokenizer.from_pretrained(MODEL_NAME)
    _preview_ids  = _tok_preview.encode(INPUT_TEXT)
    _preview_strs = [_tok_preview.decode([t]) for t in _preview_ids]
    print(f"Input:   {INPUT_TEXT!r}")
    print(f"Tokens:  {_preview_strs}")
    print(f"SEQ_LEN: {len(_preview_ids)}")
    print(f"Total decode calls: {len(_preview_ids)} tokens x {N_LAYERS} layers = {len(_preview_ids) * N_LAYERS}")
    del _tok_preview, _preview_ids, _preview_strs

Input:   'The quick brown fox jumps'
Tokens:  ['The', ' quick', ' brown', ' fox', ' jumps']
SEQ_LEN: 5
Total decode calls: 5 tokens x 12 layers = 60


## API & extractor initialization

In [4]:
api      = HypoSpaceAPI(config=DecoderConfig(
    top_k=TOP_K,
    runtime=RuntimeConfig(device="cpu", cache_dir=CACHE_DIR),
))
canvas   = SemanticCanvas()
analyzer = MechanisticAnalyzer()
print("HypoSpaceAPI + MechanisticAnalyzer ready")

if NNSIGHT_AVAILABLE:
    tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
    extractor  = NNSightExtractor(MODEL_NAME, device="cpu", cache_dir=CACHE_DIR)
    token_ids  = tokenizer.encode(INPUT_TEXT)
    TOKEN_STRS = [tokenizer.decode([t]) for t in token_ids]
    SEQ_LEN    = len(TOKEN_STRS)
    print(f"Input: {INPUT_TEXT!r}  |  Tokens: {list(enumerate(TOKEN_STRS))}")
    print(f"Total (token x layer) pairs: {SEQ_LEN * N_LAYERS}")

HypoSpaceAPI + MechanisticAnalyzer ready


Input: 'The quick brown fox jumps'  |  Tokens: [(0, 'The'), (1, ' quick'), (2, ' brown'), (3, ' fox'), (4, ' jumps')]
Total (token x layer) pairs: 60


## Section 1 — Multi-layer, multi-token extraction

`extract_layers(inputs, layer_paths, token_index)` runs one forward pass and captures all 12 layers
at one token position. Loop over `token_index` → `SEQ_LEN` passes total, each yielding 12 vectors.
Activations cache under `.hypo_cache/nnsight/`; re-runs are instantaneous.

In [5]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    print(f"Extracting: {SEQ_LEN} forward passes, {N_LAYERS} layers each\n")
    raw_acts = {}
    for tok_idx in range(SEQ_LEN):
        raw_acts[tok_idx] = extractor.extract_layers(
            INPUT_TEXT, LAYER_PATHS, token_index=tok_idx,
        )
        dims = len(next(iter(raw_acts[tok_idx].values())))
        print(f"  token {tok_idx} {TOKEN_STRS[tok_idx]!r:14s}: {N_LAYERS} layers x {dims} dims")
    print(f"\nTotal activation vectors: {SEQ_LEN * N_LAYERS}")

Extracting: 5 forward passes, 12 layers each

  token 0 'The'         : 12 layers x 768 dims
  token 1 ' quick'      : 12 layers x 768 dims
  token 2 ' brown'      : 12 layers x 768 dims
  token 3 ' fox'        : 12 layers x 768 dims
  token 4 ' jumps'      : 12 layers x 768 dims

Total activation vectors: 60


## Section 2 — Decode + mechanistic intervention

Merged loop: one `api.decode()` + `analyzer.run_interventions()` call per (token, layer) pair.
`source_index` = the neuron dimension index in [0, 767] — the **cluster key**.

In [6]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="No prior kernel found.*")
        records = []
        for tok_idx in range(SEQ_LEN):
            for layer_idx, (lp, lid) in enumerate(zip(LAYER_PATHS, LAYER_IDS)):
                decode_result = api.decode(MODEL_NAME, lid, raw_acts[tok_idx][lp], version=VERSION)
                interventions = analyzer.run_interventions(decode_result.features)
                effect_by_id  = {iv.feature_id: iv.effect_size for iv in interventions}
                for feat in decode_result.features:
                    intensity = feat.label.split()[0] if feat.label else "unknown"
                    records.append({
                        "token_idx":    tok_idx,
                        "token_str":    TOKEN_STRS[tok_idx],
                        "layer_idx":    layer_idx,
                        "layer_path":   lp,
                        "feature_id":   feat.id,
                        "source_index": feat.source_index,
                        "score":        feat.score,
                        "label":        feat.label or "unknown",
                        "intensity":    intensity,
                        "effect_size":  effect_by_id.get(feat.id, 0.0),
                    })
    df = pd.DataFrame(records)
    print(f"Records: {len(df)}  |  Unique neuron dims: {df['source_index'].nunique()}")
    print(df["intensity"].value_counts().to_string())

Records: 480  |  Unique neuron dims: 32
intensity
low-intensity       246
medium-intensity    128
high-intensity      106


## Section 3 — 3D semantic & mechanistic concept map

X=token, Y=layer, Z=score. Color=intensity band. Size=effect size. Hover for full detail.

In [7]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    traces = []
    for band, color in INTENSITY_COLORS.items():
        sub = df[df["intensity"] == band]
        if sub.empty:
            continue
        hover = (
            "<b>" + sub["feature_id"] + "</b><br>"
            + "label: " + sub["label"] + "<br>"
            + "neuron dim: " + sub["source_index"].astype(str) + "<br>"
            + "layer: " + sub["layer_path"] + "<br>"
            + "token: " + sub["token_str"].astype(str)
            + " (pos " + sub["token_idx"].astype(str) + ")<br>"
            + "score: " + sub["score"].round(4).astype(str)
        )
        traces.append(go.Scatter3d(
            x=sub["token_idx"], y=sub["layer_idx"], z=sub["score"],
            mode="markers", name=band,
            marker=dict(size=4 + sub["effect_size"] * 20, color=color,
                        opacity=0.75, line=dict(width=0.5, color="white")),
            text=hover, hovertemplate="%{text}<extra></extra>",
        ))
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=f'GPT-2 3D Semantic Concept Map - "{INPUT_TEXT}"',
        scene=dict(
            xaxis=dict(title="Token Position", tickvals=list(range(SEQ_LEN)), ticktext=TOKEN_STRS),
            yaxis=dict(title="Layer Depth", tickvals=list(range(N_LAYERS)), ticktext=LAYER_LABELS),
            zaxis=dict(title="Feature Score"),
        ),
        legend_title="Intensity Band", height=700,
    )
    fig.show()

## Section 4 — Neuron cluster identification

Group all feature records by `source_index`. Each group = one neuron across all (token, layer) positions.
Rank by `total_score` (sum of scores) — rewards both frequency and magnitude.

In [8]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    cluster_stats = []
    for src_idx, group in df.groupby("source_index"):
        traj = (group.groupby("token_idx")["score"]
                .max().reindex(range(SEQ_LEN), fill_value=0.0).tolist())
        cluster_stats.append({
            "source_index":       int(src_idx),
            "n_positions":        len(group),
            "mean_score":         round(float(group["score"].mean()), 4),
            "max_score":          round(float(group["score"].max()), 4),
            "total_score":        round(float(group["score"].sum()), 4),
            "mean_effect_size":   round(float(group["effect_size"].mean()), 4),
            "layer_spread":       int(group["layer_idx"].nunique()),
            "layers_active":      sorted(int(x) for x in group["layer_idx"].unique()),
            "score_trajectory":   traj,
            "dominant_intensity": group["intensity"].mode().iloc[0],
        })
    clusters_df  = pd.DataFrame(cluster_stats).sort_values("total_score", ascending=False).reset_index(drop=True)
    top_clusters = clusters_df.head(TOP_N_CLUSTERS).reset_index(drop=True)
    print(f"Total unique neuron clusters: {len(clusters_df)}")
    cols = ["source_index","n_positions","mean_score","max_score","layer_spread","dominant_intensity"]
    print(top_clusters[cols].to_string(index=False))

Total unique neuron clusters: 32
 source_index  n_positions  mean_score  max_score  layer_spread dominant_intensity
          447           60      0.9453     1.0000            12     high-intensity
          373           60      0.6500     1.0000            12     high-intensity
           64           48      0.6209     1.0000            12   medium-intensity
          266           47      0.4614     0.8391            12   medium-intensity
          326           46      0.3605     0.6315            11      low-intensity
          481           30      0.5524     0.9584             6     high-intensity
          288           33      0.3114     0.5385            10      low-intensity
          756           33      0.1957     0.3915            11      low-intensity
          496           10      0.4954     1.0000             2      low-intensity
          138           11      0.4043     1.0000            11      low-intensity


In [9]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    row_bg = [INTENSITY_COLORS.get(v, "#ecf0f1") for v in top_clusters["dominant_intensity"]]
    fig = go.Figure(go.Table(
        header=dict(values=["Neuron Dim","# Positions","Mean Score","Max Score","Layer Spread","Dominant Intensity"],
                    fill_color="steelblue", font=dict(color="white", size=13), align="center"),
        cells=dict(values=[top_clusters["source_index"], top_clusters["n_positions"],
                            top_clusters["mean_score"].round(4), top_clusters["max_score"].round(4),
                            top_clusters["layer_spread"], top_clusters["dominant_intensity"]],
                   fill_color=[row_bg] * 6, align="center", font=dict(size=12)),
    ))
    fig.update_layout(title=f"Top {TOP_N_CLUSTERS} Neuron Clusters — ranked by total activation", height=420)
    fig.show()

## Section 5 — Temporal evolution: cluster score trajectories

In [10]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    fig = go.Figure()
    for _, crow in top_clusters.iterrows():
        src   = int(crow["source_index"])
        traj  = crow["score_trajectory"]
        color = INTENSITY_COLORS.get(crow["dominant_intensity"], "#95a5a6")
        fig.add_trace(go.Scatter(
            x=list(range(SEQ_LEN)), y=traj, mode="lines+markers",
            name=f"dim {src}", line=dict(color=color, width=2), marker=dict(size=7),
            hovertemplate=f"<b>Neuron dim {src}</b><br>token: %{{x}}<br>score: %{{y:.4f}}<extra></extra>",
        ))
    fig.update_layout(
        title=f'Neuron Cluster Trajectories — top {TOP_N_CLUSTERS}<br><sup>"{INPUT_TEXT}"</sup>',
        xaxis=dict(title="Token Position", tickvals=list(range(SEQ_LEN)), ticktext=TOKEN_STRS),
        yaxis=dict(title="Max Score Across Layers"),
        legend_title="Neuron Dimension", height=520,
    )
    fig.show()

## Section 6 — 3D cluster trajectories

Connected lines in (token × layer × score) space. "Deep-firing" = many layers, one token.
"Broad-firing" = many tokens, similar layer depths.

In [11]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    fig = go.Figure()
    for _, crow in top_clusters.iterrows():
        src = int(crow["source_index"])
        sub = df[df["source_index"] == src].sort_values(["token_idx","layer_idx"])
        fig.add_trace(go.Scatter3d(
            x=sub["token_idx"], y=sub["layer_idx"], z=sub["score"],
            mode="lines+markers", name=f"dim {src}",
            marker=dict(size=5, opacity=0.9), line=dict(width=3),
            hovertemplate=f"<b>dim {src}</b><br>token: %{{x}}<br>layer: %{{y}}<br>score: %{{z:.4f}}<extra></extra>",
        ))
    fig.update_layout(
        title=f"3D Cluster Trajectories — top {TOP_N_CLUSTERS}",
        scene=dict(
            xaxis=dict(title="Token Position", tickvals=list(range(SEQ_LEN)), ticktext=TOKEN_STRS),
            yaxis=dict(title="Layer Depth", tickvals=list(range(N_LAYERS)), ticktext=LAYER_LABELS),
            zaxis=dict(title="Feature Score"),
        ),
        height=650,
    )
    fig.show()

## Section 7 — Interactive cluster isolation (dropdown → heatmap)

In [12]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    import ipywidgets as widgets
    from IPython.display import display

    cluster_options = [
        (f"dim {int(r['source_index'])} - {r['dominant_intensity']} ({int(r['n_positions'])} positions)",
         int(r["source_index"]))
        for _, r in top_clusters.iterrows()
    ]
    dropdown = widgets.Dropdown(options=cluster_options, description="Cluster:",
                                layout=widgets.Layout(width="70%"))
    output   = widgets.Output()

    def _render_cluster(src):
        with output:
            output.clear_output(wait=True)
            sub   = df[df["source_index"] == src]
            pivot = sub.pivot_table(index="layer_idx", columns="token_idx",
                                    values="score", aggfunc="max", fill_value=0.0)
            pivot = pivot.reindex(range(N_LAYERS), fill_value=0.0).reindex(columns=range(SEQ_LEN), fill_value=0.0)
            fig = go.Figure(go.Heatmap(
                z=pivot.values, x=[TOKEN_STRS[i] for i in range(SEQ_LEN)], y=LAYER_LABELS,
                colorscale="Plasma", text=pivot.values.round(4), texttemplate="%{text}",
                hovertemplate="layer: %{y}<br>token: %{x}<br>score: %{z:.4f}<extra></extra>",
                colorbar=dict(title="Score"),
            ))
            fig.update_layout(title=f"Neuron dim {src} — score heatmap (layer x token)",
                              xaxis_title="Token", yaxis_title="Layer", height=420)
            fig.show()
            detail = (sub[["token_str","layer_path","feature_id","score","effect_size","label"]]
                      .sort_values("score", ascending=False).reset_index(drop=True))
            print(detail.to_string(index=False))

    dropdown.observe(lambda c: _render_cluster(c["new"]), names="value")
    display(dropdown, output)
    _render_cluster(int(top_clusters.iloc[0]["source_index"]))

Dropdown(description='Cluster:', layout=Layout(width='70%'), options=(('dim 447 - high-intensity (60 positions…

Output()

## Section 8 — Layer spread per token

In [13]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    top_dim_set = set(top_clusters["source_index"].astype(int).tolist())
    plot_data = []
    for tok_idx in range(SEQ_LEN):
        tok_sub = df[(df["token_idx"] == tok_idx) & df["source_index"].isin(top_dim_set)]
        for src, grp in tok_sub.groupby("source_index"):
            plot_data.append({"token_str": TOKEN_STRS[tok_idx],
                               "source_index": str(int(src)),
                               "layer_count": int(grp["layer_idx"].nunique())})
    cdf = pd.DataFrame(plot_data)
    if not cdf.empty:
        fig = px.bar(cdf, x="token_str", y="layer_count", color="source_index", barmode="group",
                     title=f"Layer Spread per Token — top {TOP_N_CLUSTERS} clusters",
                     labels={"token_str":"Token","layer_count":"# Layers Active","source_index":"Neuron Dim"},
                     height=440)
        fig.show()

## Section 9 — Per-layer feature score distributions

In [14]:
if not NNSIGHT_AVAILABLE:
    print("Skipping - nnsight not available")
else:
    fig = make_subplots(rows=2, cols=6, subplot_titles=LAYER_LABELS,
                        shared_xaxes=True, shared_yaxes=True)
    for layer_idx in range(N_LAYERS):
        sub = df[df["layer_idx"] == layer_idx]
        fig.add_trace(
            go.Histogram(x=sub["score"], nbinsx=20, marker_color="#3498db",
                         opacity=0.75, showlegend=False),
            row=layer_idx // 6 + 1, col=layer_idx % 6 + 1,
        )
    fig.update_layout(title_text="Feature Score Distributions Across All 12 Layers", height=500)
    fig.show()

## Section 10 — Automatic Semantic Labeling

Decode each neuron cluster from a raw dimension index to a human-readable semantic concept:

1. **wte vocabulary scoring** — Column `d` of GPT-2's embedding matrix `wte [50257×768]` scores
   every vocabulary token by how much it "lives" in dimension `d`. Top tokens = concept footprint.
2. **WordNet Lowest Common Hypernym (LCH)** — Most specific shared ancestor in the IS-A hierarchy.
   `"red","blue","green"` → `"chromatic color"`. Polysemantic neurons bubble to generic labels.
3. **SBERT OOV fallback** — Tokens not in WordNet: embed the list with `all-MiniLM-L6-v2`,
   nearest-neighbor search against pre-embedded WordNet noun lemmas.
4. **UMAP + HDBSCAN** — Embed each cluster's concept centroid, reduce to 2D, auto-group.

In [15]:
if not (NNSIGHT_AVAILABLE and SEMANTIC_AVAILABLE):
    print("Skipping semantic pipeline - requires nnsight + nltk + sentence-transformers + umap-learn + hdbscan")
else:
    sbert     = SentenceTransformer(SBERT_MODEL)
    _ref_lemmas = list(set(n.replace("_", " ") for n in list(wn.all_lemma_names(pos="n"))[:3000]))
    _ref_embs   = sbert.encode(_ref_lemmas, convert_to_numpy=True, show_progress_bar=False)
    print(f"SBERT ready: {SBERT_MODEL}")
    print(f"WordNet reference: {len(_ref_lemmas)} noun lemmas pre-embedded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SBERT ready: all-MiniLM-L6-v2
WordNet reference: 3000 noun lemmas pre-embedded


In [16]:
if not (NNSIGHT_AVAILABLE and SEMANTIC_AVAILABLE):
    print("Skipping")
else:
    from transformers import GPT2Model

    _gpt2      = GPT2Model.from_pretrained(MODEL_NAME)
    wte_weight = _gpt2.wte.weight.detach().numpy()
    del _gpt2

    _all_strs   = [tokenizer.decode([i]) for i in range(tokenizer.vocab_size)]
    _clean_ids  = [i for i, s in enumerate(_all_strs) if s.strip().isalpha() and len(s.strip()) >= 3]
    _clean_strs = [_all_strs[i].strip().lower() for i in _clean_ids]
    wte_clean   = wte_weight[_clean_ids, :]

    top_vocab_tokens = {}
    for src_idx in clusters_df["source_index"]:
        col   = wte_clean[:, int(src_idx)]
        top_k = np.argsort(np.abs(col))[-WTE_TOP_N:][::-1]
        top_vocab_tokens[int(src_idx)] = [_clean_strs[i] for i in top_k]

    print(f"Scored {len(top_vocab_tokens)} dims against {len(_clean_ids)} clean vocab tokens")
    for src_idx in list(top_vocab_tokens)[:3]:
        print(f"  dim {src_idx}: {top_vocab_tokens[src_idx][:8]}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Scored 32 dims against 44666 clean vocab tokens
  dim 447: ['guiactiveun', 'cloneembedreportprint', 'randomredditor', 'externalto', 'reportprint', 'thenitrome', 'oreandonline', 'サーティ']
  dim 373: ['reportprint', 'rawdownload', 'thenitrome', 'randomredditor', 'embedreportprint', '龍契士', 'サーティ', 'externaltoeva']
  dim 64: ['dragonmagazine', 'estream', 'ciating', 'ãâãâ', 'uliffe', '龍契士', 'cloneembedreportprint', 'oreandonline']


In [17]:
def _wn_label(tokens):
    synsets = []
    for tok in tokens[:15]:
        syns = wn.synsets(tok)
        if syns:
            synsets.append(syns[0])
    if len(synsets) < 2:
        return None
    lch = synsets[0]
    for s in synsets[1:]:
        cands = lch.lowest_common_hypernyms(s)
        if cands:
            lch = cands[0]
    name = lch.lemmas()[0].name().replace("_", " ")
    if name in ("entity", "physical entity", "abstraction", "matter", "thing", "object"):
        return None
    return name

def _sbert_label(tokens):
    emb  = sbert.encode(tokens[:10], convert_to_numpy=True).mean(axis=0)
    norm = np.linalg.norm(_ref_embs, axis=1) * np.linalg.norm(emb) + 1e-8
    sims = (_ref_embs @ emb) / norm
    return _ref_lemmas[int(sims.argmax())]

if not (NNSIGHT_AVAILABLE and SEMANTIC_AVAILABLE):
    print("Skipping")
else:
    semantic_label = {}
    for src_idx, tokens in top_vocab_tokens.items():
        label = _wn_label(tokens) or _sbert_label(tokens)
        semantic_label[int(src_idx)] = label

    clusters_df["semantic_label"] = clusters_df["source_index"].map(semantic_label).fillna("unknown")
    df["semantic_label"]           = df["source_index"].map(semantic_label).fillna("unknown")

    print("Semantic labels assigned:")
    print(clusters_df[["source_index","semantic_label","total_score"]].head(15).to_string(index=False))

Semantic labels assigned:
 source_index semantic_label  total_score
          447       external      56.7201
          373       outcaste      39.0025
           64       aperient      29.8032
          266         adagio      21.6872
          326        israeli      16.5813
          481       external      16.5724
          288         audile      10.2751
          756            net       6.4580
          496       vagabond       4.9542
          138        aliquot       4.4474
           99           hoar       3.8003
          480      cosponsor       3.0493
           87       rhomboid       2.3446
          635         audile       1.7082
          459           naif       1.3218


In [18]:
if not (NNSIGHT_AVAILABLE and SEMANTIC_AVAILABLE):
    print("Skipping")
else:
    dim = sbert.get_sentence_embedding_dimension()
    centroid_matrix = np.zeros((len(clusters_df), dim))
    for i, src_idx in enumerate(clusters_df["source_index"]):
        tokens = top_vocab_tokens.get(int(src_idx), ["unknown"])
        centroid_matrix[i] = sbert.encode(tokens[:15], convert_to_numpy=True).mean(axis=0)

    reducer       = umap_lib.UMAP(n_components=2, random_state=42, min_dist=0.1)
    umap_xy       = reducer.fit_transform(centroid_matrix)
    clusterer     = hdbscan_lib.HDBSCAN(min_cluster_size=2, min_samples=1)
    cluster_groups = clusterer.fit_predict(umap_xy)

    clusters_df["umap_x"]        = umap_xy[:, 0]
    clusters_df["umap_y"]        = umap_xy[:, 1]
    clusters_df["cluster_group"] = cluster_groups

    n_groups = len(set(cluster_groups)) - (1 if -1 in cluster_groups else 0)
    print(f"UMAP + HDBSCAN: {n_groups} semantic groups found")
    print(clusters_df[["source_index","semantic_label","cluster_group"]].head(10).to_string(index=False))

/tmp/ipykernel_12272/3771066228.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = sbert.get_sentence_embedding_dimension()


/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP + HDBSCAN: 9 semantic groups found
 source_index semantic_label  cluster_group
          447       external              0
          373       outcaste              0
           64       aperient              0
          266         adagio              1
          326        israeli              7
          481       external              0
          288         audile              3
          756            net              7
          496       vagabond              1
          138        aliquot              4


## Section 11 — UMAP Semantic Atlas

Every neuron cluster as a point in 2D semantic space. Color = HDBSCAN group.
Size = total activation. Label = auto-detected semantic concept.
Hover shows the top vocabulary tokens that activate this neuron.

In [19]:
if not (NNSIGHT_AVAILABLE and SEMANTIC_AVAILABLE):
    print("Skipping")
else:
    fig = go.Figure()
    for gid in sorted(clusters_df["cluster_group"].unique()):
        sub   = clusters_df[clusters_df["cluster_group"] == gid]
        color = CLUSTER_PALETTE[gid % len(CLUSTER_PALETTE)] if gid >= 0 else "#aaaaaa"
        hover = sub.apply(
            lambda r: (f"<b>dim {int(r.source_index)}</b><br>"
                       f"label: {r.semantic_label}<br>"
                       f"score: {r.total_score:.3f}<br>"
                       f"top tokens: {top_vocab_tokens.get(int(r.source_index), [])[:6]}"),
            axis=1,
        )
        fig.add_trace(go.Scatter(
            x=sub["umap_x"], y=sub["umap_y"],
            mode="markers+text",
            name=f"group {gid}" if gid >= 0 else "singleton",
            text=sub["semantic_label"], textposition="top center",
            marker=dict(size=8 + sub["total_score"] * 6, color=color, opacity=0.85,
                        line=dict(width=1, color="white")),
            hovertext=hover, hoverinfo="text",
        ))
    fig.update_layout(
        title="UMAP Semantic Atlas — GPT-2 Residual Stream Concept Space",
        xaxis_title="UMAP-1", yaxis_title="UMAP-2",
        height=600,
    )
    fig.show()

## Section 12 — Semantic × Layer / Token × Semantic Heatmaps

**Left**: Which semantic concepts dominate at each transformer layer depth?
**Right**: Which concepts does each input token activate?

In [20]:
if not (NNSIGHT_AVAILABLE and SEMANTIC_AVAILABLE):
    print("Skipping")
else:
    # Semantic x Layer
    pivot_sl = (df.groupby(["semantic_label","layer_idx"])["score"]
                .mean().unstack("layer_idx").fillna(0.0))
    fig = go.Figure(go.Heatmap(
        z=pivot_sl.values, x=LAYER_LABELS, y=pivot_sl.index.tolist(),
        colorscale="Viridis",
        hovertemplate="concept: %{y}<br>layer: %{x}<br>mean score: %{z:.4f}<extra></extra>",
        colorbar=dict(title="Mean Score"),
    ))
    fig.update_layout(
        title="Semantic Concept × Layer — mean activation per depth",
        xaxis_title="Layer", yaxis_title="Semantic Concept",
        height=max(350, len(pivot_sl) * 28 + 100),
    )
    fig.show()

In [21]:
if not (NNSIGHT_AVAILABLE and SEMANTIC_AVAILABLE):
    print("Skipping")
else:
    # Token x Semantic
    ordered_tokens = [TOKEN_STRS[i] for i in range(SEQ_LEN)]
    pivot_ts = (df.groupby(["token_str","semantic_label"])["score"]
                .max().unstack("semantic_label").fillna(0.0)
                .reindex(ordered_tokens))
    fig = go.Figure(go.Heatmap(
        z=pivot_ts.values, x=pivot_ts.columns.tolist(), y=ordered_tokens,
        colorscale="Plasma",
        text=pivot_ts.values.round(3), texttemplate="%{text}",
        hovertemplate="token: %{y}<br>concept: %{x}<br>max score: %{z:.4f}<extra></extra>",
        colorbar=dict(title="Max Score"),
    ))
    fig.update_layout(
        title='Token × Semantic Concept — "what does each word activate?"',
        xaxis_title="Semantic Concept", yaxis_title="Token",
        height=max(300, SEQ_LEN * 40 + 100),
    )
    fig.show()

## Section 13 — Semantic Composition Flow

Stacked area chart: how the proportion of activation belonging to each semantic concept
shifts from early (h.0) to late (h.11) transformer layers.

In [22]:
if not (NNSIGHT_AVAILABLE and SEMANTIC_AVAILABLE):
    print("Skipping")
else:
    layer_totals = (df.groupby(["layer_idx","semantic_label"])["score"]
                    .mean().unstack("semantic_label").fillna(0.0))
    layer_pct    = layer_totals.div(layer_totals.sum(axis=1).replace(0, 1), axis=0)

    fig = go.Figure()
    for label in layer_pct.columns:
        fig.add_trace(go.Scatter(
            x=LAYER_LABELS, y=layer_pct[label],
            mode="lines", stackgroup="one", name=label, fill="tonexty",
            hovertemplate=f"<b>{label}</b><br>layer: %{{x}}<br>share: %{{y:.1%}}<extra></extra>",
        ))
    fig.update_layout(
        title="Semantic Composition Flow — concept proportions across depth",
        xaxis_title="Layer", yaxis_title="Proportion of Activation",
        yaxis_tickformat=".0%", height=480,
    )
    fig.show()

## Section 14 — Semantic Cluster Co-activation Network

Nodes = unique semantic labels. Edge weight = fraction of (token, layer) positions
where both labels co-activate. Layout uses SVD of the co-activation matrix as a
spring-force proxy — no external graph library needed.

In [23]:
if not (NNSIGHT_AVAILABLE and SEMANTIC_AVAILABLE):
    print("Skipping")
else:
    labels_list = sorted(df["semantic_label"].unique())
    n_labels    = len(labels_list)
    label_idx   = {l: i for i, l in enumerate(labels_list)}

    coact = np.zeros((n_labels, n_labels))
    for (tok, lay), grp in df.groupby(["token_idx","layer_idx"]):
        active = grp["semantic_label"].unique()
        for a in active:
            for b in active:
                if a != b:
                    coact[label_idx[a], label_idx[b]] += 1

    coact_norm = coact / (coact.max() or 1)

    # SVD layout (spring proxy — no networkx needed)
    U, S, _ = np.linalg.svd(coact_norm)
    x_pos   = U[:, 0] * S[0]
    y_pos   = U[:, 1] * S[1] if len(S) > 1 else np.zeros(n_labels)

    edge_traces = []
    for i in range(n_labels):
        for j in range(i + 1, n_labels):
            w = coact_norm[i, j]
            if w < 0.1:
                continue
            edge_traces.append(go.Scatter(
                x=[x_pos[i], x_pos[j], None], y=[y_pos[i], y_pos[j], None],
                mode="lines", line=dict(width=w * 8, color="rgba(100,100,200,0.4)"),
                showlegend=False, hoverinfo="skip",
            ))

    node_trace = go.Scatter(
        x=x_pos, y=y_pos, mode="markers+text",
        text=labels_list, textposition="top center",
        marker=dict(size=14, color="#3498db", line=dict(width=1.5, color="white")),
        hovertext=[f"<b>{l}</b><br>co-act strength: {coact_norm[label_idx[l]].sum():.2f}"
                   for l in labels_list],
        hoverinfo="text", showlegend=False,
    )
    fig = go.Figure(data=edge_traces + [node_trace])
    fig.update_layout(
        title="Semantic Cluster Co-activation Network",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        height=520,
    )
    fig.show()

## Section 15 — 3D Bipartite Semantic Graph

The main deliverable. Two planes connected by meaning:

- **Z=0 plane** — Token nodes, left-to-right by sequence position
- **Z=3 plane** — Semantic cluster nodes, positioned by UMAP coordinates
- **Blue edges** — Token → Semantic: which concept does each word activate?
- **Green edges** — Token ↔ Token: which tokens share active clusters (Jaccard)?
- **Red edges** — Semantic ↔ Semantic: which concepts co-activate?

Use the controls to adjust edge density or filter to a single transformer layer.

In [24]:
if not (NNSIGHT_AVAILABLE and SEMANTIC_AVAILABLE):
    print("Skipping - requires both nnsight and semantic deps")
else:
    import ipywidgets as widgets
    from IPython.display import display

    # --- Precompute node positions ---
    top_src_set = set(top_clusters["source_index"].astype(int).tolist())
    top_cl      = clusters_df[clusters_df["source_index"].isin(top_src_set)].copy()

    ux = top_cl.set_index("source_index")["umap_x"]
    uy = top_cl.set_index("source_index")["umap_y"]
    _ux_r = ux.max() - ux.min() + 1e-6
    _uy_r = uy.max() - uy.min() + 1e-6
    sem_x = {int(s): (ux[s] - ux.min()) / _ux_r * 6 - 3 for s in ux.index}
    sem_y = {int(s): (uy[s] - uy.min()) / _uy_r * 6 - 3 for s in uy.index}
    tok_x = {i: (i - (SEQ_LEN - 1) / 2) * 2.0 for i in range(SEQ_LEN)}

    # --- Precompute edge weights ---
    tok_sem_edges = []
    for tok_idx in range(SEQ_LEN):
        for src_idx in top_src_set:
            sub = df[(df["token_idx"] == tok_idx) & (df["source_index"] == src_idx)]
            if not sub.empty:
                tok_sem_edges.append((tok_idx, int(src_idx), float(sub["score"].mean())))

    tok_clusters = {i: set(df[df["token_idx"] == i]["source_index"].unique()) & top_src_set
                    for i in range(SEQ_LEN)}
    tok_tok_edges = []
    for i in range(SEQ_LEN):
        for j in range(i + 1, SEQ_LEN):
            inter = tok_clusters[i] & tok_clusters[j]
            union = tok_clusters[i] | tok_clusters[j]
            if union:
                w = len(inter) / len(union)
                if w > 0:
                    tok_tok_edges.append((i, j, w))

    src_list = list(top_src_set)
    sem_sem_edges = []
    for ii, si in enumerate(src_list):
        for jj, sj in enumerate(src_list):
            if jj <= ii:
                continue
            pos_i = set(zip(df[df["source_index"] == si]["token_idx"],
                            df[df["source_index"] == si]["layer_idx"]))
            pos_j = set(zip(df[df["source_index"] == sj]["token_idx"],
                            df[df["source_index"] == sj]["layer_idx"]))
            w = len(pos_i & pos_j) / max(len(pos_i | pos_j), 1)
            if w > 0:
                sem_sem_edges.append((si, sj, w))

    print(f"Edge counts — tok→sem: {len(tok_sem_edges)}, tok↔tok: {len(tok_tok_edges)}, sem↔sem: {len(sem_sem_edges)}")

    # --- Graph builder ---
    def _build_graph(threshold, show_ts, show_tt, show_ss, layer_filter):
        traces = []

        grp_colors = [CLUSTER_PALETTE[int(g) % len(CLUSTER_PALETTE)] if g >= 0 else "#aaaaaa"
                      for g in top_cl["cluster_group"]]
        traces.append(go.Scatter3d(
            x=[sem_x.get(int(s), 0) for s in top_cl["source_index"]],
            y=[sem_y.get(int(s), 0) for s in top_cl["source_index"]],
            z=[3.0] * len(top_cl),
            mode="markers+text", name="Semantic Clusters",
            text=top_cl["semantic_label"].tolist(), textposition="top center",
            marker=dict(size=10, color=grp_colors, opacity=0.9, line=dict(width=1, color="white")),
            hovertext=[f"<b>dim {int(r.source_index)}</b><br>{r.semantic_label}<br>"
                       f"top tokens: {top_vocab_tokens.get(int(r.source_index), [])[:5]}"
                       for _, r in top_cl.iterrows()],
            hoverinfo="text",
        ))
        traces.append(go.Scatter3d(
            x=[tok_x[i] for i in range(SEQ_LEN)], y=[0.0] * SEQ_LEN, z=[0.0] * SEQ_LEN,
            mode="markers+text", name="Tokens",
            text=TOKEN_STRS, textposition="bottom center",
            marker=dict(size=12, color="#ecf0f1", opacity=1.0, line=dict(width=2, color="#2c3e50")),
            hovertext=[f"<b>{t}</b> (pos {i})" for i, t in enumerate(TOKEN_STRS)],
            hoverinfo="text",
        ))

        if show_ts:
            xs, ys, zs = [], [], []
            for (ti, si, w) in tok_sem_edges:
                if w < threshold:
                    continue
                if layer_filter >= 0:
                    sub = df[(df["token_idx"] == ti) & (df["source_index"] == si)
                             & (df["layer_idx"] == layer_filter)]
                    if sub.empty:
                        continue
                xs += [tok_x[ti], sem_x.get(si, 0), None]
                ys += [0.0, sem_y.get(si, 0), None]
                zs += [0.0, 3.0, None]
            if xs:
                traces.append(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                                           name="Token→Semantic",
                                           line=dict(color="rgba(52,152,219,0.35)", width=2),
                                           hoverinfo="skip"))

        if show_tt:
            xs, ys, zs = [], [], []
            for (i, j, w) in tok_tok_edges:
                if w < threshold:
                    continue
                mx = (tok_x[i] + tok_x[j]) / 2
                xs += [tok_x[i], mx, tok_x[j], None]
                ys += [0.0, 0.3, 0.0, None]
                zs += [0.0, 0.25, 0.0, None]
            if xs:
                traces.append(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                                           name="Token↔Token",
                                           line=dict(color="rgba(46,204,113,0.5)", width=3),
                                           hoverinfo="skip"))

        if show_ss:
            xs, ys, zs = [], [], []
            for (si, sj, w) in sem_sem_edges:
                if w < threshold:
                    continue
                mx = (sem_x.get(si, 0) + sem_x.get(sj, 0)) / 2
                my = (sem_y.get(si, 0) + sem_y.get(sj, 0)) / 2
                xs += [sem_x.get(si, 0), mx, sem_x.get(sj, 0), None]
                ys += [sem_y.get(si, 0), my, sem_y.get(sj, 0), None]
                zs += [3.0, 3.4, 3.0, None]
            if xs:
                traces.append(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines",
                                           name="Semantic↔Semantic",
                                           line=dict(color="rgba(231,76,60,0.45)", width=2),
                                           hoverinfo="skip"))

        fig = go.Figure(data=traces)
        fig.update_layout(
            title=f'3D Bipartite Semantic Graph — "{INPUT_TEXT}"',
            scene=dict(
                xaxis=dict(title="Position / UMAP-1", showgrid=False),
                yaxis=dict(title="", showgrid=False, showticklabels=False),
                zaxis=dict(title="Plane", tickvals=[0, 3], ticktext=["Tokens", "Semantics"]),
                camera=dict(eye=dict(x=1.5, y=-1.5, z=1.2)),
            ),
            legend_title="Node / Edge Type", height=750,
        )
        return fig

    # --- Widget controls ---
    thresh_slider = widgets.FloatSlider(value=EDGE_THRESHOLD, min=0.0, max=1.0, step=0.05,
                                        description="Edge threshold:",
                                        style={"description_width": "initial"},
                                        layout=widgets.Layout(width="50%"))
    ts_check = widgets.Checkbox(value=True, description="Token→Semantic (blue)")
    tt_check = widgets.Checkbox(value=True, description="Token↔Token (green)")
    ss_check = widgets.Checkbox(value=True, description="Semantic↔Semantic (red)")
    layer_drop = widgets.Dropdown(
        options=[("All layers", -1)] + [(f"Layer {i} ({LAYER_LABELS[i]})", i) for i in range(N_LAYERS)],
        value=-1, description="Layer filter:", style={"description_width": "initial"},
    )
    graph_out = widgets.Output()

    def _refresh(*_):
        with graph_out:
            graph_out.clear_output(wait=True)
            _build_graph(thresh_slider.value, ts_check.value, tt_check.value,
                         ss_check.value, layer_drop.value).show()

    for w in [thresh_slider, ts_check, tt_check, ss_check, layer_drop]:
        w.observe(_refresh, names="value")

    display(widgets.VBox([widgets.HBox([thresh_slider, layer_drop]),
                          widgets.HBox([ts_check, tt_check, ss_check])]),
            graph_out)
    _refresh()

Edge counts — tok→sem: 42, tok↔tok: 10, sem↔sem: 41


Output()

## Section 16 — Diagnostics & summary

In [25]:
report = run_diagnostics()
print(f"Diagnostics - overall: {report.overall_status}")
for probe in report.probes:
    icon = {"ok": "OK  ", "degraded": "WARN", "error": "ERR "}.get(probe.status, "?   ")
    print(f"  [{icon}] {probe.subsystem:32s} ({probe.latency_ms:.1f} ms)")

print()
if NNSIGHT_AVAILABLE:
    print("Notebook summary:")
    print(f"  Input text:           {INPUT_TEXT!r}")
    print(f"  Tokens:               {SEQ_LEN}")
    print(f"  Layers:               {N_LAYERS}")
    print(f"  Feature records:      {len(df)}")
    print(f"  Unique neuron dims:   {df['source_index'].nunique()}")
    print(f"  Neuron clusters:      {len(clusters_df)}")
    print(f"  Top clusters shown:   {TOP_N_CLUSTERS}")
    print(f"  Forward passes:       {SEQ_LEN} (one per token position)")
    if SEMANTIC_AVAILABLE and "semantic_label" in clusters_df.columns:
        print(f"  Semantic labels:      {clusters_df['semantic_label'].nunique()} unique")
        print(f"  HDBSCAN groups:       {clusters_df['cluster_group'].nunique()}")
else:
    print("nnsight not available - install torch + nnsight")

/home/user/HypoSpace/api.py:109: UserWarning: No prior kernel found for 'diag-model-diag-layer'; cross-run match rate set to 0.0
  result = self.decoder.decode(model_name=model_name, layer=layer, activations=values, version=version)


Diagnostics - overall: degraded
  [OK  ] config                           (0.1 ms)
  [OK  ] preprocessor                     (0.1 ms)
  [OK  ] hierarchy                        (0.1 ms)
  [OK  ] kernel_library                   (1.8 ms)
  [OK  ] extractor                        (10.1 ms)
  [OK  ] semantic                         (0.1 ms)
  [WARN] mechanistic                      (0.1 ms)
  [OK  ] full_pipeline                    (16.0 ms)
  [OK  ] governance                       (0.1 ms)
  [OK  ] nnsight                          (216.5 ms)
  [?   ] pyvene                           (0.1 ms)
  [OK  ] sae_backend                      (0.3 ms)

Notebook summary:
  Input text:           'The quick brown fox jumps'
  Tokens:               5
  Layers:               12
  Feature records:      480
  Unique neuron dims:   32
  Neuron clusters:      32
  Top clusters shown:   10
  Forward passes:       5 (one per token position)
  Semantic labels:      30 unique
  HDBSCAN groups:       10
